# Agentic RAG — wired onto your own `RAGBase`

This builds directly on `rag_helper.py` + `ingest.py` from your `01-intro` module.
We're translating Lessons 13/14 from the **Responses API** shape (OpenAI-only)
into the **Chat Completions** shape (Groq, and later Ollama) — same concept,
different wire format.

**Before running anything:** fix `rag_helper.py` line 64 —
`def llm(self, prompt):` has 3 spaces of indent, every other method has 4.
Make it 4 spaces or this import will throw `IndentationError`.


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.3-70b-versatile"  # same model your RAGBase already defaults to


## 1. Reuse your existing index + RAGBase

Nothing new here — same `ingest.py` + `rag_helper.py` you already built.
The agent will call `rag_system.search(...)`, the exact same method
`RAGBase.rag()` already uses internally.


In [ ]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

rag_system = RAGBase(index=index, llm_client=groq_client, course="llm-zoomcamp", model=MODEL)

# sanity check: does the old fixed pipeline still work?
print(rag_system.rag("is it too late to join the course?"))


## 2. Define the tool schema (Chat Completions shape)

Compare this to the Responses-API schema from Lesson 13 — same fields
(`name`, `description`, `parameters`), but nested one level deeper under
`"function"`. This is just wire-format, not a new concept.

The `description` field is still the most important line in this whole
cell — it's the only thing the model reads to decide *when* to call `search`.


In [ ]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


## 3. The dispatch helper

Turns a `tool_call` object from Groq's response into an actual function
execution against `rag_system.search()`, then packages the result as a
Chat-Completions-style `tool` message.

This is filled in for you — it's mechanical glue, same idea as Lesson 14's
`make_call`, just shaped for Chat Completions instead of Responses.


In [ ]:
def make_tool_call(tool_call, rag_system):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = rag_system.search(**args)
    else:
        result = {"error": f"unknown tool: {tool_call.function.name}"}

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    }


## 4. YOUR TURN — the agent loop

This is the part you build yourself. Don't copy Lesson 14's code directly —
that's Responses-API shape. Build the Chat Completions version using what
you already know:

**The three parts (Lesson 14's "anatomy of an agent"):**
- `instructions` → goes in as a `{"role": "system", ...}` message (Groq doesn't
  use the `"developer"` role from the lesson — use `"system"`, same idea)
- `tools` → `[search_tool]`, passed to every `groq_client.chat.completions.create()` call
- `memory` → the growing `messages` list — same pattern you already drew out
  by hand earlier in this session

**The loop condition** (this is the one thing to get exactly right):
- Call the model, get back `response.choices[0].message`
- Append that message object to `messages` (the model needs to see its own decision)
- Check `message.tool_calls` — if it's **not empty**: for each tool_call, run
  `make_tool_call()` and append the result to `messages`. Loop again.
- If `message.tool_calls` is **empty/None**: that's the final answer. Stop.

Fill in the `while True:` block below. Use `it` to print iteration numbers
like the lesson does, so you can watch it loop in real time.


In [ ]:
def agent_loop(instructions, question, model=MODEL, max_iterations=5):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1
    last_answer = None

    while True:
        print(f"iteration #{it}...")

        # TODO 1: call groq_client.chat.completions.create(
        #             model=model, messages=messages, tools=[search_tool]
        #         )

        # TODO 2: grab the message: response.choices[0].message
        #         append it to `messages`

        # TODO 3: check message.tool_calls
        #         - if present: loop over them, call make_tool_call(),
        #           append each result to `messages`, set a flag
        #         - if message.content exists, that's a candidate answer —
        #           save it to last_answer

        # TODO 4: increment `it`
        # TODO 5: break the while loop if there were no tool_calls this round
        #         (also break if it > max_iterations, as a safety net —
        #          the lesson mentions this as a guard real frameworks add)

        break  # remove this once your loop logic is in place

    return last_answer


## 5. Test it — the typo case

Once your loop is filled in, run this. Watch the iteration count: a working
agent should search "Olama", get nothing useful, then search "Ollama" on its
own and find the answer — without you writing any typo-handling code.


In [ ]:
instructions = '''
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches if the first results are not satisfying.
'''.strip()

answer = agent_loop(instructions, "How do I run Olama locally?")
print()
print("FINAL ANSWER:")
print(answer)


## 6. Bridge to the hospital project — a guardrail instruction

Earlier you started sketching this in German: an instruction that keeps the
agent scoped to hospital data and tells it what to do when it's *not*
confident, rather than guessing with sensitive data.

Write your own version below — one instructions string, then test it with
a question that's clearly out of scope (the chess example from Lesson 14:
`"what's queen gambit?"`) and confirm the agent refuses instead of answering
from general knowledge.


In [ ]:
hospital_style_instructions = '''
# TODO: write your own scoped + guardrail instructions here,
# modeled on Lesson 14's off-topic restriction, but for a hospital
# FAQ/SAP context instead of a course FAQ.
'''.strip()

# test with an off-topic question once you've written the instructions above
# answer = agent_loop(hospital_style_instructions, "what's queen gambit?")
# print(answer)
